# APD hinge publication figure trial

Existing Hugh APD records; Richard's 23–30 September display proposals. This is a reproducible figure trial, not a final inference of DNA states or absolute hinge angles. See `review.html` for the email chronology and figure requirements.

The original notebooks and Python helpers were copied into `original_scripts`; originals remain unchanged. The native 4 µs maps count **every valid sample**, not a display-thinned cloud. Circle fitting and long time traces use **100-sample (0.4 ms) means of channels**, with every complete block retained. That is a deliberate trial setting, different from fitting raw native angles; compare fit stability before publication.

## Conventions and corrections

- File channels `[90,45,135,0]` become matrix order `[0,90,45,135]`. Fit inverse relative gains, then apply Hugh's already-inverted optical matrix. No extra inversion, background fitting, centering or stretching.
- Default: manuscript annular model, NA 1.3, inner NA 0.39, water n=1.33. The source viewer instead defaults to annular+Fresnel with inner NA 0.38 and oil n=1.515. Both are compared; these are model estimates.
- Invalid angles remain NaN. No artificial mass is clipped onto 90°. Native validity fractions are reported; every displayed probability is conditional on validity.
- Optical θ is from the laboratory vertical. φ is modulo π. A low-occupancy cut chooses one representative quarter-sphere, not a unique physical trajectory.
- Stereographic radius **ρ=2 tan(θ/2)**; `(u,v)=2(x,y)/(1+z)`. Thus `dΩ=du dv/(1+ρ²/4)²`. Each square bin's solid angle is integrated by quadrature; density is `count/(N_valid ΔΩ)`. Bins crossing the branch/support boundary include the zero-density outside portion, so avoid interpreting individual edge bins as exact pointwise estimates.
- Hinge coordinates are a rigid rotation after choosing the laboratory branch. `x′ ∝ n×z` is horizontal; `y′=n×x′`. β=`atan2(p·y′,p·x′)`. A rod-to-arm offset is unmeasured, so β is not automatically the hinge opening α.

Richard's proposed four-degeneracy orbit-constrained assignment is **not** applied by default: it can compress points toward the assumed circle. A small circle residual is not independent evidence of correct reconstruction.


In [ ]:
%matplotlib inline
import json
import numpy as np
import matplotlib.pyplot as plt
import paper_figures as pf
from IPython.display import display, Image

pf.MODEL = 'manuscript_annular'  # or 'notebook_fresnel'; rerun subsequent cells
FILES = ['1.tdms', '3.tdms', '34.tdms']
records = [pf.process_record(name) for name in FILES]


## Stereographic maps and θ profiles

Maps use a shared colour scale in sr⁻¹ in both coordinate systems. The right-hand profile is the probability in a θ shell divided by its exact solid angle `π(cos θ_low−cos θ_high)` on the folded quadrant. It is a shell average, not the ordinary probability density per degree. An isotropic folded distribution is `1/π sr⁻¹`.

The camera-like integration used for the θ profile can hide fast motion; native maps and nested windows expose that distinction. The hinge view is cropped for legibility; its included mass is in the summary, and normalization always uses all valid samples.

In [ ]:
display(pf.overview(records)); plt.close('all')

## Cut placement and raw-count audit

In [ ]:
display(pf.cut_figure(records[0])); plt.close('all')

## Laboratory geometry and rotation about the fitted axis

In [ ]:
display(pf.geometry_figure(records)); plt.close('all')

## Nested native time windows

For a repeatable first look, the centre is the largest adjacent change between valid 0.4 ms θ estimates in the first five seconds. This is a display-selection rule, not a transition detector or dwell-time measurement. The shortest panel shows original 4 µs samples. Lines join consecutive valid observations; NaNs break them. The orange overlay averages **intensities before angle reconstruction**.

In [ ]:
for r in records:
    display(pf.zoom_figure(r)); plt.close('all')

## Optical-model sensitivity

In [ ]:
display(pf.model_figure(records)); plt.close('all')

## Interactive cut and native-window inspection

The cut slider operates on the 0.4 ms display cloud only. It does not change the stored maps, fit or raw data. The native window control reloads unaveraged data from disk. The plotted circle is a model summary, not an imposed correction.

In [ ]:
import ipywidgets as widgets
def inspect_cut(file='1.tdms', cut_deg=0.):
    r=next(r for r in records if r['name']==file)
    ph=pf.wrap_phi(r['ph'][r['ok']],np.radians(cut_deg))
    pts=pf.stereo(pf.vectors(r['th'][r['ok']],ph))
    fig,ax=plt.subplots(figsize=(5,4));ax.scatter(*pts[::5].T,s=.25)
    ax.set(xlim=(-2.1,2.1),ylim=(-2.1,2.1),aspect='equal',xlabel='u',ylabel='v',title=f'cut={cut_deg:.1f}°; display thinned only')
    plt.show()
widgets.interact(inspect_cut,file=FILES,cut_deg=widgets.FloatSlider(value=np.degrees(records[0]['cut']),min=0,max=180,step=.5))


In [ ]:
def inspect_native(file='1.tdms', start_s=.1825, width_us=600.):
    r=next(r for r in records if r['name']==file)
    t,a=pf.native_window(pf.DATA/file,r['g'],start_s,start_s+width_us/1e6)
    th,ph,ok,x,y=pf.angles(a)
    xy=pf.stereo(pf.vectors(th,pf.wrap_phi(ph,r['cut'])))
    fig,axs=plt.subplots(1,2,figsize=(9,3.4))
    axs[0].plot((t-start_s)*1e6,np.degrees(th),'.-',ms=2,lw=.6)
    axs[0].set(xlabel='time from start (µs)',ylabel='optical θ (°)')
    axs[1].plot(*xy.T,'.-',ms=2,lw=.6);axs[1].set(aspect='equal',xlabel='u',ylabel='v')
    plt.show()
widgets.interact(inspect_native,file=FILES,start_s=widgets.FloatText(value=.1825),width_us=widgets.FloatText(value=600.))


In [ ]:
summary=[r['summary'] for r in records]
(pf.ROOT/'apd_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
summary
